# 04 — Character detection
Same DASS preprocessing, NMS and face/body pairing as 1pageTest.

Run detection inference with the CUDA ONNX provider. This notebook does not train the detector. 
CPU remains available for graph operators unsupported by CUDA; model initialization must succeed on CUDA, 
and runtime fallback to a CPU-only session is disabled. Image preprocessing and box cleanup use the CPU.

Cached results remain reusable. Set `FORCE_RERUN = True` to recompute existing pages on the GPU. 
Restart the kernel and rerun setup before using this change, so an older CPU session is not reused.


Use .venvAppliedTask. Load the ONNX session only for an uncached page.

In [5]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if not (ROOT / "project.json").is_file():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import pipeline_helpers as P
import json
from PIL import Image, ImageDraw, ImageFont
ROOT = P.ROOT
MODELS = P.read_json(ROOT / "models.json")
SPLITS = ["train", "val"]  # Or ["train"] / ["val"].
FORCE_RERUN = False
items = P.pages(SPLITS)
print("Pages:", len(items))
import math
import numpy as np
import onnxruntime as ort
MODEL = Path(MODELS["character_detection"]["local_path"])
CONFIDENCE = 0.35
NMS_IOU = 0.45
if not MODEL.is_file():
    raise FileNotFoundError(MODEL)
GPU_DEVICE_ID = 0  # First NVIDIA GPU.
session = None


Pages: 240


Cache faces, bodies and character candidates separately for each page.

In [6]:
for item in items:
    PAGE = Path(item["image_path"])
    page = Image.open(PAGE).convert("RGB")
    OUT = P.page_folder(item)
    OUT.mkdir(parents=True, exist_ok=True)
    settings = {"confidence":CONFIDENCE, "nms_iou":NMS_IOU, "size":1024, "version":1}
    dependencies = [MODEL]
    def save_stage(name, records):
        P.save_stage(item, name, records, settings, dependencies)
    def load_stage(name):
        return P.load_stage(item, name)
    if all(P.cached(item, name, settings, dependencies, FORCE_RERUN) for name in ['faces.json', 'bodies.json', 'characters.json']):
        print("Cached:", item["split"], item["sequence_id"], item["page_number"])
        continue
    if session is None:
        available = ort.get_available_providers()
        print("Available ONNX providers:", available)
        if "CUDAExecutionProvider" not in available:
            raise RuntimeError(
                "CUDA is unavailable in this kernel's ONNX Runtime. "
                "Use an environment with onnxruntime-gpu and compatible CUDA/cuDNN libraries."
            )
        session = ort.InferenceSession(
            str(MODEL),
            providers=[("CUDAExecutionProvider", {"device_id":GPU_DEVICE_ID}),
                       "CPUExecutionProvider"],
        )
        active = session.get_providers()
        if not active or active[0] != "CUDAExecutionProvider":
            session = None
            raise RuntimeError(
                "The CUDA provider failed to initialize. Check the ONNX Runtime log "
                "for missing CUDA/cuDNN libraries before continuing."
            )
        session.disable_fallback()  # Fail instead of switching to CPU-only inference.
        print(f"Character detector uses CUDA GPU {GPU_DEVICE_ID}.")
        print("Active ONNX providers:", active)
    size = 1024
    scale = min(size / page.width, size / page.height)
    rw, rh = max(1, round(page.width * scale)), max(1, round(page.height * scale))
    left, top = (size - rw) // 2, (size - rh) // 2
    canvas = Image.new("RGB", (size, size), (114, 114, 114))
    canvas.paste(page.resize((rw, rh), Image.Resampling.BILINEAR), (left, top))
    tensor = np.asarray(canvas, dtype=np.float32).transpose(2, 0, 1)[None]
    faces_raw, bodies_raw = session.run(None, {session.get_inputs()[0].name: tensor})
    print("Providers:", session.get_providers())
    print("Faces:", faces_raw.shape, "Bodies:", bodies_raw.shape)

    def decode_head(raw):
        rows = np.asarray(raw)[0]
        if rows.ndim != 2 or rows.shape[1] != 5:
            raise ValueError(f"Expected [1, N, 5] detections; got {raw.shape}")
        rows = rows[np.isfinite(rows).all(axis=1) & (rows[:, 4] >= CONFIDENCE)]
        boxes = np.column_stack((rows[:, 0] - rows[:, 2] / 2, rows[:, 1] - rows[:, 3] / 2,
                                 rows[:, 0] + rows[:, 2] / 2, rows[:, 1] + rows[:, 3] / 2))
        order = np.argsort(-rows[:, 4], kind="stable")
        keep = []
        while len(order):
            index = int(order[0])
            keep.append(index)
            rest = order[1:]
            intersection = np.maximum(0, np.minimum(boxes[index, 2:], boxes[rest, 2:]) -
                                       np.maximum(boxes[index, :2], boxes[rest, :2])).prod(axis=1)
            area = np.maximum(0, boxes[:, 2:] - boxes[:, :2]).prod(axis=1)
            iou = intersection / np.maximum(area[index] + area[rest] - intersection, 1e-6)
            order = rest[iou <= NMS_IOU]
        detections = []
        for index in keep:
            box = boxes[index].copy()
            box[[0, 2]] = (box[[0, 2]] - left) / (rw / page.width)
            box[[1, 3]] = (box[[1, 3]] - top) / (rh / page.height)
            x1, y1, x2, y2 = box
            box = [max(0, math.floor(x1)), max(0, math.floor(y1)),
                   min(page.width, math.ceil(x2)), min(page.height, math.ceil(y2))]
            if box[2] > box[0] and box[3] > box[1]:
                detections.append({"box": box, "confidence": float(rows[index, 4])})
        return detections

    faces, bodies = decode_head(faces_raw), decode_head(bodies_raw)
    save_stage("faces.json", faces)
    save_stage("bodies.json", bodies)

    def face_coverage(face, body):
        f, b = face["box"], body["box"]
        intersection = max(0, min(f[2], b[2]) - max(f[0], b[0])) * max(0, min(f[3], b[3]) - max(f[1], b[1]))
        return intersection / ((f[2] - f[0]) * (f[3] - f[1]))

    characters = [{"box": body["box"], "body": body, "face": None} for body in bodies]
    candidates = sorted(((face_coverage(face, body), fi, bi)
                         for fi, face in enumerate(faces) for bi, body in enumerate(bodies)), reverse=True)
    used_faces, used_bodies = set(), set()
    for coverage, fi, bi in candidates:
        if coverage >= 0.8 and fi not in used_faces and bi not in used_bodies:
            characters[bi]["face"] = faces[fi]
            used_faces.add(fi)
            used_bodies.add(bi)
    for fi, face in enumerate(faces):
        if fi not in used_faces:
            characters.append({"box": face["box"], "face": face, "body": None})
    characters.sort(key=lambda c: (c["box"][1], -c["box"][0]))
    for i, character in enumerate(characters, 1):
        character["character_id"] = f"C{i}"
    save_stage("characters.json", characters)


Cached: train seq_2032620aa4e4ac7f 1
Cached: train seq_2032620aa4e4ac7f 2
Cached: train seq_2032620aa4e4ac7f 3
Cached: train seq_952f154fb1505883 1
Cached: train seq_952f154fb1505883 2
Cached: train seq_952f154fb1505883 3
Cached: train seq_7db7a000adeceacf 1
Cached: train seq_7db7a000adeceacf 2
Cached: train seq_7db7a000adeceacf 3
Cached: train seq_9032e5551cf1a9d2 1
Cached: train seq_9032e5551cf1a9d2 2
Cached: train seq_9032e5551cf1a9d2 3
Cached: train seq_2022b8c0e1f2d3c0 1
Cached: train seq_2022b8c0e1f2d3c0 2
Cached: train seq_2022b8c0e1f2d3c0 3
Cached: train seq_a3bd36201a0cd61b 1
Cached: train seq_a3bd36201a0cd61b 2
Cached: train seq_a3bd36201a0cd61b 3
Cached: train seq_72fdfff4b4353a42 1
Cached: train seq_72fdfff4b4353a42 2
Cached: train seq_72fdfff4b4353a42 3
Cached: train seq_b251c8e1ba8b91f8 1
Cached: train seq_b251c8e1ba8b91f8 2
Cached: train seq_b251c8e1ba8b91f8 3
Cached: val seq_ee880b9612205860 1
Cached: val seq_ee880b9612205860 2
Cached: val seq_ee880b9612205860 3
Cached:

KeyboardInterrupt: 